In [2]:
import librosa
%matplotlib inline
import matplotlib.pyplot as plt
import librosa.display
from IPython.display import Audio
import numpy as np
import tensorflow as tf
from matplotlib.pyplot import specgram
import pandas as pd
from sklearn.metrics import confusion_matrix
import IPython.display as ipd  # To play sound in the notebook
import os # interface with underlying OS that python is running on
import sys

In [5]:
# LOAD IN FILE
#main_dir = 'C:/Laptop3/Makaleler/makale19 speech recog/ravdes/ravdes/Audio_Speech_Actors_01-24/'
main_dir = "C:/Users/Aruay/Desktop/ra application/project/dataset/ravdes/audio_speech_actors_01-24"
#main_dir ='D:/Makaleler/makale19 speech recog/ravdes/ex/'
sub_dir = os.listdir(main_dir)
x, sr = librosa.load(main_dir+'/'+sub_dir[0]+'/03-01-02-02-01-02-02.wav')
# PLAY any AUDIO FILE
#librosa.output.write_wav('MaleNeutral.wav', x, sr)
Audio(data=x, rate=sr)

In [12]:
def extract_feature(file_name, offst=0.5):
    X, sample_rate = librosa.load(file_name, res_type='kaiser_fast',offset=offst)
    
    stft = np.abs(librosa.stft(X))
    chroma_cq = librosa.feature.chroma_cqt(y=X, sr=sample_rate)
    cqt=np.mean(chroma_cq,axis=1)
    
    #chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate,n_fft=2048).T,axis=0)
    #chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
    
    mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T,axis=0)
    
    mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate,n_mels=128,fmax=8000).T,axis=0)
    #mel = np.mean(librosa.feature.melspectrogram(X, sr=sample_rate).T,axis=0)
    
    contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate).T,axis=0)
    
    tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(X), sr=sample_rate).T,axis=0)
    return mfccs,cqt,mel,contrast,tonnetz
   

def extract_featurev2(file_name):
    X, sample_rate = librosa.load(file_name, res_type='kaiser_fast', sr=None)   
    #mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=64).T, axis=0)
    C = np.abs(librosa.cqt(X, sr=sr, fmin=librosa.note_to_hz('C2'), n_bins=60))
    #d=librosa.amplitude_to_db(C)
    d=librosa.power_to_db(C)
    cqt = np.mean(d,axis=1)
    return cqt

In [13]:
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)

In [14]:
# CREATE FUNCTION TO EXTRACT EMOTION NUMBER, ACTOR AND GENDER LABEL
emotion = []
gender = []
actor = []
file_path = []
file_pathShort = []
a= []
b= []
c= []
d= []
e= []

X4= pd.DataFrame()
cc=0
for i in sub_dir:
    filename = os.listdir(main_dir + '/' + i) #iterate over Actor folders
    #print(filename)
    for f in filename: 
        part = f.split('.')[0].split('-')
        emotion.append(int(part[2]))
        actor.append(int(part[6]))
        bg = int(part[6])
        if bg%2 == 0:
            bg = 0 #"female"
        else:
            bg = 1 #"male"
        gender.append(bg)
        file_path.append(main_dir + '/' + i + '/' + f)
        file_pathShort.append('/' + i + '/' + f)
        a,b,c,d,e=extract_feature(main_dir + '/' + i + '/' + f)
        #a=extract_feature(main_dir + i + '/' + f)
        tot= []
        for x in a:tot.append(x)
        for x in b:tot.append(x)
        for x in c:tot.append(x)
        for x in d:tot.append(x)
        for x in e:tot.append(x)
        
        #X4[cc]=pd.DataFrame([tot])
        X4[cc]=np.asarray(tot)
        cc+=1;
        #print(tot)
    
# PUT EXTRACTED LABELS WITH FILEPATH INTO DATAFRAME
audio_df = pd.DataFrame(emotion)
#audio_df = audio_df.replace({1:'neutral', 2:'calm', 3:'happy', 4:'sad', 5:'angry', 6:'fear', 7:'disgust', 8:'surprise'})
audio_df = pd.concat([pd.DataFrame(gender),audio_df,pd.DataFrame(actor)],axis=1)
audio_df.columns = ['gender','emotion','actor']
audio_df = pd.concat([audio_df,pd.DataFrame(file_pathShort, columns = ['pathShort']),
                      pd.DataFrame(file_path, columns = ['path'])],axis=1)
len(tot)

c:\Users\Aruay\anaconda3\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=1012
  warnings.warn(
c:\Users\Aruay\anaconda3\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=932
  warnings.warn(
c:\Users\Aruay\anaconda3\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=966
  warnings.warn(
C:\Users\Aruay\AppData\Local\Temp\ipykernel_13516\493552666.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X4[cc]=np.asarray(tot)
C:\Users\Aruay\AppData\Local\Temp\ipykernel_13516\493552666.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

193

In [15]:
len(a),len(b),len(c),len(d),len(e)

(40, 12, 128, 7, 6)

In [16]:
X6= pd.DataFrame(X4.T)

In [17]:
type(X6), X6.shape

(pandas.core.frame.DataFrame, (1380, 193))

In [18]:
X7=X6.copy()

In [19]:
X7

,0,1,2,3,4,5,6,7,8,9,...,183,184,185,186,187,188,189,190,191,192
0,-625.688965,66.489113,-13.966170,13.211882,-8.448891,-0.996674,-9.327506,-6.938110,-7.516921,1.637754,...,15.009738,17.080975,17.110049,39.262881,-0.038146,0.034928,0.015396,-0.062674,0.010666,0.004282
1,-622.542236,60.999710,-15.575345,16.737888,-10.040482,-0.414517,-9.070066,-5.629632,-5.936283,0.398298,...,15.671255,16.802633,16.617909,38.895488,-0.037206,0.033166,0.030498,-0.025112,0.012125,-0.008584
2,-594.578430,62.131107,-15.727082,8.557268,-8.745336,-0.615630,-9.618413,-7.869385,-8.680186,0.413947,...,15.368281,17.121289,18.026363,38.937130,-0.052715,0.049443,0.035664,-0.044645,0.002154,-0.013212
3,-612.539856,58.425236,-15.162426,9.593304,-11.289780,-2.178137,-11.070035,-8.933653,-10.518451,-0.771062,...,15.269961,16.729708,17.741164,38.253174,-0.010430,0.047339,-0.084709,-0.022829,0.016673,-0.000173
4,-639.076904,63.925850,-8.221095,17.328478,-9.261359,1.697167,-7.593100,-5.222811,-6.721763,0.477244,...,16.186734,17.552213,16.494480,38.659161,-0.088642,0.063712,-0.018844,-0.011945,0.003047,0.004860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1375,-558.012939,33.520958,-25.292545,2.852234,-22.460226,-6.676313,-22.499645,-13.656723,-11.484236,-4.583565,...,16.289653,18.007045,17.446851,39.516740,-0.019101,0.014588,0.020755,0.051313,-0.005109,0.022226
1376,-509.447357,48.729256,-22.362350,-1.398089,-22.556530,-10.647760,-17.144878,-8.896639,-14.071460,0.397057,...,16.132877,16.953431,17.792023,38.693052,-0.041246,0.017704,0.014374,0.016831,0.016845,-0.000732
1377,-517.197327,39.712631,-26.507496,2.247680,-22.035622,-14.673651,-19.214750,-9.156967,-12.672953,3.738830,...,16.041746,16.897425,18.422812,38.928571,-0.031410,0.033743,-0.049835,0.008919,0.007375,0.013000
1378,-463.539062,34.857422,-13.660451,8.858073,-19.420994,0.439111,-17.810989,-4.469375,-7.315601,0.363834,...,16.245986,17.837695,17.299424,39.203358,-0.046634,0.026736,-0.043650,-0.061472,0.006547,-0.008751


In [20]:
X7.shape

(1380, 193)

In [21]:
X8=pd.concat([X7,pd.DataFrame(gender),pd.DataFrame(emotion)],axis=1)

In [22]:
X8.head()

,0,1,2,3,4,5,6,7,8,9,...,185,186,187,188,189,190,191,192,0,0
0,-625.688965,66.489113,-13.966170,13.211882,-8.448891,-0.996674,-9.327506,-6.938110,-7.516921,1.637754,...,17.110049,39.262881,-0.038146,0.034928,0.015396,-0.062674,0.010666,0.004282,0,1
1,-622.542236,60.999710,-15.575345,16.737888,-10.040482,-0.414517,-9.070066,-5.629632,-5.936283,0.398298,...,16.617909,38.895488,-0.037206,0.033166,0.030498,-0.025112,0.012125,-0.008584,0,1
2,-594.578430,62.131107,-15.727082,8.557268,-8.745336,-0.615630,-9.618413,-7.869385,-8.680186,0.413947,...,18.026363,38.937130,-0.052715,0.049443,0.035664,-0.044645,0.002154,-0.013212,0,1
3,-612.539856,58.425236,-15.162426,9.593304,-11.289780,-2.178137,-11.070035,-8.933653,-10.518451,-0.771062,...,17.741164,38.253174,-0.010430,0.047339,-0.084709,-0.022829,0.016673,-0.000173,0,1
4,-639.076904,63.925850,-8.221095,17.328478,-9.261359,1.697167,-7.593100,-5.222811,-6.721763,0.477244,...,16.494480,38.659161,-0.088642,0.063712,-0.018844,-0.011945,0.003047,0.004860,0,2


In [23]:
X8.tail()

,0,1,2,3,4,5,6,7,8,9,...,185,186,187,188,189,190,191,192,0,0
1375,-558.012939,33.520958,-25.292545,2.852234,-22.460226,-6.676313,-22.499645,-13.656723,-11.484236,-4.583565,...,17.446851,39.516740,-0.019101,0.014588,0.020755,0.051313,-0.005109,0.022226,0,8
1376,-509.447357,48.729256,-22.362350,-1.398089,-22.556530,-10.647760,-17.144878,-8.896639,-14.071460,0.397057,...,17.792023,38.693052,-0.041246,0.017704,0.014374,0.016831,0.016845,-0.000732,0,8
1377,-517.197327,39.712631,-26.507496,2.247680,-22.035622,-14.673651,-19.214750,-9.156967,-12.672953,3.738830,...,18.422812,38.928571,-0.031410,0.033743,-0.049835,0.008919,0.007375,0.013000,0,8
1378,-463.539062,34.857422,-13.660451,8.858073,-19.420994,0.439111,-17.810989,-4.469375,-7.315601,0.363834,...,17.299424,39.203358,-0.046634,0.026736,-0.043650,-0.061472,0.006547,-0.008751,0,8
1379,-498.993683,38.967426,-7.731936,3.212382,-10.446373,-1.859776,-16.630434,-1.310444,-14.403450,2.125301,...,16.524717,39.351774,-0.066635,0.060339,-0.020905,-0.046316,-0.011483,0.018687,0,8


In [24]:
X7.to_csv("featureNormal_Ek.csv",index=False,header=True)
#audio_df.to_csv("audioELM.csv",index=False,header=True)